# Verify fixed & merged H5 file

Plot static images, postage stamps, and point source locations from the merged_cleaned H5 file,
to verify that `fix_h5_metadata.py` + `merge_h5.py` + `merge_clean_h5.py` produced correct results.

In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter

In [ ]:
%matplotlib inline

In [ ]:
MAG_ZPS_CPS = {
    'u': 26.52,
    'g': 28.51,
    'r': 28.36,
    'i': 28.17,
    'z': 27.78,
    'y': 26.82,
}

PIX2ARCSEC = 0.2

def flux2mag(flux, band, exposure_time=30.):
    return -2.5 * np.log10(flux / exposure_time) + MAG_ZPS_CPS[band]

def mag2flux(mag, band, exposure_time=30.):
    return 10**((mag - MAG_ZPS_CPS[band]) / (-2.5)) * exposure_time

In [ ]:
filename = "3000sqdeg_lsst_1y_sample_merged_cleaned.h5"
hf = h5py.File(filename, "r")
print("Number of systems:", len(hf))

## Helper functions

In [ ]:
def get_point_xy(ind, band, hf=hf):
    """Get point source pixel coords relative to stamp center."""
    metadata = hf[f'lsst_lens_{ind}']['metadata']
    n_images = len(hf[f'lsst_lens_{ind}']['light_curves'])
    x_c, y_c = 16, 16  # stamp center
    xs, ys = [], []
    for ind2 in range(n_images):
        dx = metadata.attrs[f'point_source_light_{band}_ra_image_{ind2}'] / PIX2ARCSEC
        dy = metadata.attrs[f'point_source_light_{band}_dec_image_{ind2}'] / PIX2ARCSEC
        xs.append(x_c + dx)
        ys.append(y_c + dy)
    return xs, ys


def plot_static(ind, band, hf=hf):
    """Plot the static image (host galaxy + lensed arc)."""
    arr = hf[f'lsst_lens_{ind}']['static_image'][band]['lens_plus_lensed_agn_host'][:]
    plt.figure(figsize=(4, 4))
    plt.imshow(np.arcsinh(arr), origin="lower")
    plt.colorbar()
    plt.title(f"Static image: lens {ind}, {band}-band")


def plot_stamp(ind, band, time_index=0, hf=hf):
    """Plot a postage stamp with point source locations overlaid."""
    arr = hf[f'lsst_lens_{ind}']['postage_stamps'][band]['all_observations'][time_index]
    plt.figure(figsize=(4, 4))
    plt.imshow(np.arcsinh(arr), origin="lower")
    plt.colorbar()
    xs, ys = get_point_xy(ind, band)
    plt.plot(xs, ys, 'm.', ms=8)
    plt.title(f"Stamp: lens {ind}, {band}-band, t={time_index}")


def plot_diff(ind, band, time_index=0, hf=hf):
    """Plot stamp - stamp_mean (smoothed), overlay point sources, verify flux vs light curve mag."""
    arr_static = hf[f'lsst_lens_{ind}']['static_image'][band]['lens_plus_lensed_agn_host'][:]
    arr_stamp = hf[f'lsst_lens_{ind}']['postage_stamps'][band]['all_observations'][time_index]
    arr_stamp_mean = np.mean(
        hf[f'lsst_lens_{ind}']['postage_stamps'][band]['all_observations'][:], axis=0
    )

    plt.figure(figsize=(4, 4))
    plt.imshow(
        np.arcsinh(gaussian_filter(arr_stamp - arr_stamp_mean, sigma=2)),
        origin="lower", vmin=-8, vmax=8, cmap='bwr'
    )
    plt.colorbar()

    xs, ys = get_point_xy(ind, band)
    plt.plot(xs, ys, 'm.', ms=8)
    plt.title(f"Diff (stamp - mean): lens {ind}, {band}, t={time_index}")

    # Verify: extract point source flux and compare to light curve mag
    light_curves = hf[f'lsst_lens_{ind}']['light_curves']
    point_src = arr_stamp - arr_static
    for ind2 in range(len(xs)):
        x, y = int(xs[ind2]), int(ys[ind2])
        arr_sel = point_src[y-2:y+3, x-2:x+3]
        total_flux = np.sum(arr_sel)
        lc_mag = light_curves[f'image_{ind2}'][band][time_index]
        print(f"  image_{ind2}: flux_mag={flux2mag(total_flux, band):.2f}, lc_mag={lc_mag:.2f}")

## Pick a system to inspect

In [ ]:
ind = 233
band = 'i'

n_images = len(hf[f'lsst_lens_{ind}']['light_curves'])
n_epochs = hf[f'lsst_lens_{ind}']['postage_stamps'][band]['all_observations'].shape[0]
print(f"Lens {ind}: {n_images} point source images, {n_epochs} epochs")

In [ ]:
plot_static(ind, band)

In [ ]:
plot_stamp(ind, band, time_index=0)

In [ ]:
plot_diff(ind, band, time_index=0)

In [ ]:
plot_diff(ind, band, time_index=2)

## Spot check multiple systems

Check a few systems across the merged file to verify consistency.

In [ ]:
n_total = len(hf)
check_indices = [0, 24, 100, 422, 500, 1000, 1500, 2000, n_total - 1]
check_indices = [i for i in check_indices if i < n_total]

for ind in check_indices:
    n_images = len(hf[f'lsst_lens_{ind}']['light_curves'])
    xs, ys = get_point_xy(ind, 'i')
    print(f"Lens {ind}: {n_images} images, point x={[f'{x:.1f}' for x in xs]}, y={[f'{y:.1f}' for y in ys]}")

In [ ]:
# Plot a grid of systems
fig, axs = plt.subplots(2, 4, figsize=(16, 8))
axs = axs.flatten()

for i, ind in enumerate(check_indices[:8]):
    arr = hf[f'lsst_lens_{ind}']['postage_stamps']['i']['all_observations'][0]
    axs[i].imshow(np.arcsinh(arr), origin="lower")
    xs, ys = get_point_xy(ind, 'i')
    axs[i].plot(xs, ys, 'm.', ms=8)
    axs[i].set_title(f"Lens {ind}")

plt.tight_layout()

## Close file

In [ ]:
hf.close()